# 📘 scikit-image 소개와 기본

**scikit-image**(skimage)는 과학적 이미지 처리를 위한 파이썬 라이브러리입니다.
NumPy 배열을 기반으로 하며, 연구와 산업 모두에서 널리 사용됩니다.

**scikit-image vs 다른 라이브러리:**

| 특징 | scikit-image | OpenCV | Pillow |
|------|-------------|--------|--------|
| 데이터 형식 | NumPy 배열 (RGB) | NumPy 배열 (BGR) | PIL Image |
| 설계 철학 | 과학/연구용 | 실시간 컴퓨터 비전 | 웹 이미지 편집 |
| 알고리즘 | 다양한 이미지 처리 알고리즘 | 컴퓨터 비전 특화 | 기본 편집/변환 |
| API 스타일 | 함수형 (모듈별) | 객체형 (cv2 모듈) | 객체형 (Image 클래스) |
| 서브모듈 | color, filters, transform, ... | cv2 전역 | Image, ImageDraw, ... |

**학습 목표:**
- scikit-image 설치와 기본 사용법
- 내장 샘플 이미지 활용
- 이미지 읽기/쓰기와 데이터 타입
- 색상 공간 변환

## 1. 설치와 임포트

```bash
pip install scikit-image
```

> 💡 `import skimage`가 아닌 `from skimage import ...` 형식으로 임포트합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  scikit-image 임포트와 기본 설정          │
# └─────────────────────────────────────────┘

from skimage import data, color, io, filters, transform, exposure, measure, morphology, segmentation
from skimage.util import img_as_float, img_as_ubyte, img_as_uint
import numpy as np
import matplotlib.pyplot as plt

print('scikit-image 임포트 완료!')
print(f'사용 가능한 샘플 이미지: {len([x for x in dir(data) if not x.startswith("_")])}개')

# 대표적인 샘플 이미지
sample_images = ['astronaut', 'camera', 'coins', 'page', 'chelsea', 'coffee']
print('\n=== 대표 샘플 이미지 ===')
for name in sample_images:
    img = getattr(data, name)()
    print(f'  {name:>12s}: {str(img.shape):>15s}  dtype={img.dtype}')

# matplotlib 한글 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 2. 내장 샘플 이미지

scikit-image는 연구용 샘플 이미지를 내장하고 있습니다.
`skimage.data` 모듈에서 바로 사용할 수 있습니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  내장 샘플 이미지 활용                   │
# │  data 모듈에서 바로 로드                  │
# └─────────────────────────────────────────┘

# 샘플 이미지 로드
img_astronaut = data.astronaut()  # 컬러 (512x512x3)
img_camera = data.camera()        # 흑백 (512x512)
img_coins = data.coins()          # 흑백 (303x384)
img_chelsea = data.chelsea()     # 고양이 컬러
img_coffee = data.coffee()       # 커피 컬러

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
images = [img_astronaut, img_camera, img_coins, img_chelsea, img_coffee]
titles = ['astronaut (컬러)', 'camera (흑백)', 'coins (흑백)', 'chelsea (컬러)', 'coffee (컬러)']
cmaps = [None, 'gray', 'gray', None, None]

for ax, img, title, cmap in zip(axes.flat, images, titles, cmaps):
    ax.imshow(img, cmap=cmap)
    ax.set_title(f'{title}\n{img.shape} dtype={img.dtype}')
    ax.axis('off')
axes[1, 2].axis('off')
plt.tight_layout()
plt.show()

print('💡 컬러 이미지: shape=(H, W, 3), dtype=uint8, 범위 0~255')
print('💡 흑백 이미지: shape=(H, W), dtype=uint8, 범위 0~255')
print('💡 skimage는 RGB 순서 (OpenCV의 BGR과 다름!)')

## 3. 데이터 타입과 변환

scikit-image는 다양한 데이터 타입을 지원합니다.
이미지 처리 알고리즘에 따라 적절한 타입으로 변환해야 합니다.

| 타입 | 범위 | 용도 |
|------|------|------|
| uint8 | 0~255 | 일반 이미지 |
| uint16 | 0~65535 | 의료 영상, 16비트 |
| float64 | 0.0~1.0 | 알고리즘 연산 |

In [ ]:
# ┌─────────────────────────────────────────┐
# │  데이터 타입 변환                         │
# │  uint8 <-> float64 <-> uint16            │
# └─────────────────────────────────────────┘

# 원본 이미지 (uint8, 0~255)
img = data.camera()
print(f'원본: dtype={img.dtype}, min={img.min()}, max={img.max()}')

# float64로 변환 (0.0~1.0)
img_float = img_as_float(img)
print(f'float64: dtype={img_float.dtype}, min={img_float.min():.4f}, max={img_float.max():.4f}')

# uint16으로 변환 (0~65535)
img_uint16 = img_as_uint(img)
print(f'uint16: dtype={img_uint16.dtype}, min={img_uint16.min()}, max={img_uint16.max()}')

# 다시 uint8로
img_back = img_as_ubyte(img_float)
print(f'uint8 복원: dtype={img_back.dtype}, min={img_back.min()}, max={img_back.max()}')

# 값 범위 확인
print('\n=== 값 범위 변환 규칙 ===')
print('uint8  (0~255)   -> float64 (0.0~1.0):  값/255.0')
print('float64 (0.0~1.0) -> uint8  (0~255):    값*255.0')
print('uint8  (0~255)   -> uint16 (0~65535):  값*257 (65535/255)')

# 주의: 연산 후 값 범위 클리핑
result = img_float * 1.5  # 일부 픽셀이 1.0 초과
result_clipped = np.clip(result, 0, 1)  # 클리핑 필요!
print(f'\n1.5배 연산: min={result.min():.2f}, max={result.max():.2f}')
print(f'클리핑 후: min={result_clipped.min():.2f}, max={result_clipped.max():.2f}')

# io.imread/imsave로 파일 입출력
print('\n💡 io.imread(): 이미지 파일 읽기 (자동 dtype 변환)')
print('💡 io.imsave(): 이미지 파일 저장 (float은 자동으로 0~255 범위로 변환)')

## 4. 색상 공간 변환

scikit-image의 `color` 모듈은 다양한 색상 공간 변환을 제공합니다.
OpenCV와 달리 **RGB 순서**를 사용하며, float 타입(0~1)이 기본입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  색상 공간 변환                           │
# │  RGB <-> HSV <-> Gray <-> Lab            │
# └─────────────────────────────────────────┘

img_rgb = data.astronaut()

# RGB -> 다양한 공간 변환
img_gray = color.rgb2gray(img_rgb)        # 흑백 (float, 0~1)
img_hsv = color.rgb2hsv(img_rgb)           # HSV (float, 0~1)
img_lab = color.rgb2lab(img_rgb)            # Lab (L:0~100, a/b:-128~127)
img_hed = color.rgb2hed(img_rgb)            # HED (조직 염색)

# 흑백 -> RGB (3채널 복원)
img_gray_to_rgb = color.gray2rgb(img_gray)

print(f'RGB shape: {img_rgb.shape}, dtype: {img_rgb.dtype}')
print(f'Gray shape: {img_gray.shape}, dtype: {img_gray.dtype}')
print(f'HSV shape: {img_hsv.shape}, dtype: {img_hsv.dtype}')
print(f'Lab shape: {img_lab.shape}, dtype: {img_lab.dtype}')

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(img_rgb); axes[0, 0].set_title('RGB (원본)')
axes[0, 1].imshow(img_gray, cmap='gray'); axes[0, 1].set_title('Gray (흑백)')
axes[0, 2].imshow(img_hsv[:,:,0], cmap='hsv'); axes[0, 2].set_title('HSV — Hue 채널')
axes[1, 0].imshow(img_hsv[:,:,1], cmap='gray'); axes[1, 0].set_title('HSV — Saturation')
axes[1, 1].imshow(img_hsv[:,:,2], cmap='gray'); axes[1, 1].set_title('HSV — Value')
axes[1, 2].imshow(img_lab[:,:,0], cmap='gray'); axes[1, 2].set_title('Lab — L 채널')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('💡 rgb2gray(): float(0~1) 반환, uint8로 변환하려면 img_as_ubyte()')
print('💡 rgb2hsv(): float(0~1), H=색상, S=채도, V=명도')
print('💡 rgb2lab(): Lab 공간은 지각적으로 균일한 색상 표현')

## 🎯 연습 문제

1. `data.astronaut()` 이미지를 흑백으로 변환한 후, 다시 RGB로 변환하여 원본과 비교하세요.
2. 세 가지 샘플 이미지(`camera`, `coins`, `chelsea`)를 로드하고 shape/dtype/min/max를 출력하세요.
3. HSV 이미지에서 Hue 채널이 0.1~0.3(노란색~초록색)인 픽셀의 마스크를 만드세요.
4. float64 이미지에 0.5를 곱한 후 uint8로 변환할 때, 클리핑 없이 올바른 결과를 얻으려면?
5. Lab 색상 공간에서 L 채널(밝기)만 조작해 이미지를 밝게 만드세요.